In [158]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [159]:
df = pd.read_csv('../data/01_raw/bank_transactions_data_edited.csv')

In [160]:
df.isnull().sum()

TransactionID              29
AccountID                  21
TransactionAmount          26
TransactionDate            28
TransactionType            30
Location                   30
DeviceID                   30
IP Address                 20
MerchantID                 23
Channel                    27
CustomerAge                18
CustomerOccupation         23
TransactionDuration        26
LoginAttempts              21
AccountBalance             27
PreviousTransactionDate    24
dtype: int64

In [161]:
df.duplicated().sum()

np.int64(21)

In [162]:
df.drop_duplicates()

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70.0,Doctor,81.0,1.0,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68.0,Doctor,141.0,1.0,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19.0,Student,56.0,1.0,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26.0,Student,25.0,1.0,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,NaN,Student,198.0,1.0,7429.40,2024-11-04 08:06:39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2511,TX002512,AC00009,243.08,2023-02-14 16:21:23,Credit,Jacksonville,D000215,59.127.135.25,M041,Online,24.0,Student,93.0,1.0,131.25,2024-11-04 08:07:49
2513,NaN,AC00304,313.69,2023-03-10 16:35:33,Debit,Oklahoma City,D000211,53.131.194.183,M034,Branch,73.0,Retired,152.0,1.0,7093.68,2024-11-04 08:10:23
2523,TX001691,AC00442,12.18,2023-04-20 18:50:39,Debit,New York,D000326,190.152.148.249,M088,Branch,76.0,Retired,77.0,1.0,4909.24,2024-11-04 08:07:37
2524,TX000076,AC00239,232.12,2023-12-28 17:31:03,Debit,Omaha,D000073,156.173.170.140,M066,ATM,37.0,Engineer,51.0,1.0,6689.87,2024-11-04 08:09:17


In [163]:
df.drop(['TransactionID', 'AccountID', 'DeviceID', 'IP Address', 'MerchantID'], axis=1, inplace=True)

In [164]:
df.fillna({ 'PreviousTransactionDate': df['TransactionDate'] }, inplace=True)
df.dropna(subset=['TransactionDate'], inplace=True)

In [165]:
for col in ['TransactionAmount', 'CustomerAge', 'TransactionDuration', 'LoginAttempts', 'AccountBalance']:
  median_value = df[col].median()
  df.fillna({ col: median_value }, inplace=True)

for col in ['Location', 'Channel', 'CustomerOccupation', 'TransactionType']:
  mode_value = df[col].mode()[0]
  df.fillna({ col: mode_value }, inplace=True)

df.isnull().sum()

TransactionAmount          0
TransactionDate            0
TransactionType            0
Location                   0
Channel                    0
CustomerAge                0
CustomerOccupation         0
TransactionDuration        0
LoginAttempts              0
AccountBalance             0
PreviousTransactionDate    0
dtype: int64

In [166]:
numerical_cols = df.select_dtypes(include=np.number).columns

for col in numerical_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  
  if col in ['TransactionAmount', 'AccountBalance', 'TransactionDuration']:
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])

  elif col in ['LoginAttempts']:
    pass

  elif col in ['CustomerAge']:
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

In [167]:
age_bins = [0, 29, 59, 150]
age_labels = ['Young', 'Adult', 'Older']

df['AgeGroup'] = pd.cut(df['CustomerAge'], bins=age_bins, labels=age_labels, right=True)

amount_bins = [0, 250, 1000, 5000, float('inf')]
amount_labels = ['Low', 'Medium', 'High', 'Very High']

df['AmountCategory'] = pd.cut(df['TransactionAmount'], bins=amount_bins, labels=amount_labels, right=True)

df_encoded = df.copy()

le = LabelEncoder()
df_encoded['AgeGroup_Encoded'] = le.fit_transform(df_encoded['AgeGroup'])
df_encoded['AmountCategory_Encoded'] = le.fit_transform(df_encoded['AmountCategory'])

df_encoded.drop(['AgeGroup', 'AmountCategory', 'CustomerAge', 'TransactionAmount'], axis=1, inplace=True)

In [168]:
categorical_cols = df_encoded.select_dtypes(include='object').columns

for col in categorical_cols:
  le = LabelEncoder()
  df_encoded[col] = le.fit_transform(df_encoded[col])

df_encoded.head()

,TransactionDate,TransactionType,Location,Channel,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,AgeGroup_Encoded,AmountCategory_Encoded
0,680,1,36,0,0,81.0,1.0,5112.21,129,1,0
1,1178,1,15,0,0,141.0,1.0,13758.91,216,1,1
2,1262,1,23,2,3,56.0,1.0,1122.35,65,2,0
3,818,1,33,2,3,25.0,1.0,8569.06,187,2,0
4,1939,0,1,2,3,198.0,1.0,7429.40,40,0,0


In [169]:
numerical_cols = df_encoded.select_dtypes(include=np.number).columns

scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

df_encoded.head()

,TransactionDate,TransactionType,Location,Channel,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,AgeGroup_Encoded,AmountCategory_Encoded
0,-0.782920,0.533564,1.232819,-1.226183,-1.340244,-0.550806,-0.204927,-0.002100,-0.695733,0.182405,-0.877862
1,-0.089092,0.533564,-0.488098,-1.226183,-1.340244,0.308078,-0.204927,2.227746,0.139512,0.182405,1.139131
2,0.027940,0.533564,0.167490,1.260877,1.306141,-0.908674,-0.204927,-1.031021,-1.310167,1.349893,-0.877862
3,-0.590654,0.533564,0.986973,1.260877,1.306141,-1.352431,-0.204927,0.889367,-0.138903,1.349893,-0.877862
4,0.971156,-1.874190,-1.635375,1.260877,1.306141,1.124017,-0.204927,0.595467,-1.550180,-0.985082,-0.877862


In [170]:
df_encoded.columns.tolist()

['TransactionDate',
 'TransactionType',
 'Location',
 'Channel',
 'CustomerOccupation',
 'TransactionDuration',
 'LoginAttempts',
 'AccountBalance',
 'PreviousTransactionDate',
 'AgeGroup_Encoded',
 'AmountCategory_Encoded']

In [171]:
df_encoded.to_csv('../data/02_processed/bank_transactions_data_processed.csv', index=False)